# SMART TIME — Gate 4G GPU Evidence

Run the existing Qwen3-4B + LoRA adapter against the versioned 12-case Gate 4G suite. Do not retrain. This notebook produces the prediction and score artifacts required to mark Gate 4G PASS.


In [ ]:
!rm -rf /kaggle/working/SMART-TIME-
!git clone -b feat/v3-next https://github.com/msm147852/SMART-TIME-.git /kaggle/working/SMART-TIME-
%cd /kaggle/working/SMART-TIME-
!git checkout f44d45abba8c61d573e524d177937b309d1f23f5
!git lfs install
!git lfs pull
!git rev-parse HEAD


In [ ]:
!nvidia-smi
!python - <<'PY'
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'CUDA GPU required for Gate 4G evidence'
PY


In [ ]:
!pip install -q -r infra/smart-ai/training/requirements.txt
!python -m pip check


In [ ]:
!python - <<'PY'
import json
from pathlib import Path
p=Path('backend/ai/training/smart-time-eval-v4g.jsonl')
rows=[json.loads(x) for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
print('Gate 4G cases:', len(rows))
assert len(rows)==12
assert Path('infra/smart-ai/training/output/adapter_config.json').exists()
assert Path('infra/smart-ai/training/output/adapter_model.safetensors').exists()
PY


In [ ]:
MODEL=/kaggle/working/SMART-TIME-/infra/smart-ai/training/output \
BASE_MODEL=Qwen/Qwen3-4B \
EVAL_OUTPUT=/kaggle/working/gate4g-predictions.jsonl \
python infra/smart-ai/training/evaluate_smart_ai_v4g.py


In [ ]:
!node scripts/score-smart-ai-eval-v4g.mjs /kaggle/working/gate4g-predictions.jsonl


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/gate4g-predictions.jsonl')
assert p.exists(), 'prediction artifact missing'
print(p)
print(p.read_text(encoding='utf-8'))


## PASS rule

Gate 4G PASS requires the scorer to report `pass_rate >= 0.80` with valid JSON predictions. If it fails, classify failures before any retraining. Do not claim PASS from this notebook unless the scorer itself exits successfully.
